# 03 · Join Sofascore + Capology — Germany Bundesliga 25/26 (snapshot 20260428)

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2025/26 de la Bundesliga alemana**.

⚠️ **Nota sobre el snapshot:** la temporada 25/26 está aún en curso. Se trabaja con
una foto fija de Sofascore (`df_germany_2526_snapshot_20260428.csv`). Este notebook
deberá reejecutarse con los datos definitivos cuando finalice la liga, generando
entonces el master sin sufijo de fecha (`master_germany_2526.csv`).

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_germany_2526_snapshot_20260428.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_germany_2526.csv').copy()

print(f'Sofascore (snapshot):  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:              {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore (snapshot):  494 jugadores | 117 columnas
Capology:              556 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   1 fc heidenheim
   1 fc koln
   1 fc union berlin
   1 fsv mainz 05
   bayer 04 leverkusen
   borussia m gladbach
   fc augsburg
   fc bayern munchen
   fc st pauli
   hamburger sv
   rb leipzig
   sc freiburg
   sv werder bremen
   tsg hoffenheim
   vfb stuttgart
   vfl wolfsburg

En Capology pero no en Sofascore:
   augsburg
   bayer leverkusen
   bayern munich
   freiburg
   hamburg
   heidenheim
   hoffenheim
   koln
   leipzig
   mainz
   monchengladbach
   st pauli
   stuttgart
   union berlin
   werder bremen
   wolfsburg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'augsburg':'fc augsburg',
            'bayer leverkusen':'bayer 04 leverkusen',
            'bayern munich':'fc bayern munchen',
            'freiburg':'sc freiburg',
            'hamburg':'hamburger sv',
            'heidenheim':'1 fc heidenheim',
            'hoffenheim':'tsg hoffenheim',
            'koln':'1 fc koln',
            'leipzig':'rb leipzig',
            'mainz':'1 fsv mainz 05',
            'monchengladbach':'borussia m gladbach',
            'st pauli':'fc st pauli',
            'stuttgart':'vfb stuttgart',
            'union berlin':'1 fc union berlin',
            'werder bremen':'sv werder bremen',
            'wolfsburg':'vfl wolfsburg'
            

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 441/494 (89.3%)
Sin emparejar: 53


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          4
Revisión media    (0.75 ≤ score < 0.90):   8
Revisión estricta (0.50 ≤ score < 0.75):   21
Revisión muy est. (score < 0.50):           20


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
12,Yannick Engelhardt,Borussia M'gladbach,yannik engelhardt,0.971
28,Albert Grønbæk,Hamburger SV,albert grnbaek,0.923
21,Joakim Mæhle,VfL Wolfsburg,joakim maehle,0.917
25,Lasse Riess,1. FSV Mainz 05,lasse rie,0.900


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
9,Jan Uwe Thielmann,1. FC Köln,jan thielmann,0.867
20,Victor Okoh Boniface,SV Werder Bremen,victor boniface,0.857
4,Daniel Fernandes,Hamburger SV,daniel heuer fernandes,0.842
0,Dan Zagadou,VfB Stuttgart,dan axel zagadou,0.815
45,Jonas Adjei Adjetey,VfL Wolfsburg,jonas adjetey,0.812
26,Maximilian Rosenfelder,SC Freiburg,max rosenfelder,0.811
10,Ísak Bergmann Jóhannesson,1. FC Köln,isak johannesson,0.780
37,Omar Haktab Traore,1. FC Heidenheim,omar traore,0.759


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 8 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
5,Jeff Chabot,VfB Stuttgart,julian chabot,0.667
34,Tom Alexander Rothe,1. FC Union Berlin,tom rothe,0.643
3,Kim Min-jae,FC Bayern München,min jae kim,0.636
36,Aiman Dardari,FC Augsburg,fabian rieder,0.615
24,David Daiber,FC Bayern München,konrad laimer,0.560
46,Aurelio Buta,Eintracht Frankfurt,aurele amenda,0.560
23,Jonah Kusi-Asare,FC Bayern München,jamal musiala,0.552
35,Emir Sahiti,Hamburger SV,miro muheim,0.545
15,Arne Maier,FC Augsburg,felix meiser,0.545
41,Cenny Neumann,1. FC Köln,jan thielmann,0.538


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['jeff chabot',
                    'tom alexander rothe',
                    'kim min jae'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 3


### 7.4 Revisión muy estricta (score < 0.50)

Candidatos con muy baja similitud. Por defecto ninguno se acepta.
Añadir a `ACCEPT_VERY_LOW_FUZZY` los que se confirmen manualmente.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
27,Guilherme Ramos,Hamburger SV,daniel heuer fernandes,0.486
6,Luca Reggiani,Borussia Dortmund,silas ostrzinski,0.483
33,Oladapo Afolayan,FC St. Pauli,adam dzwigala,0.483
13,Samuele Inacio,Borussia Dortmund,ramy bensebaini,0.483
40,Merlin Röhl,SC Freiburg,maximilian philipp,0.483
43,Xavi Simons,RB Leipzig,andrija maksimovic,0.483
48,Isak Hansen Aarøen,SV Werder Bremen,niklas stark,0.483
52,Kevin Kampl,RB Leipzig,david raum,0.476
7,Pascal Groß,Borussia Dortmund,salih ozcan,0.476
19,Silas,1. FSV Mainz 05,niklas tauer,0.471


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 456/494 (92.3%)
Sin salario:     38


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 38


,player,team,minutesPlayed,appearances,goals,assists
0,Leo Scienza,1. FC Heidenheim,180,2,1,0
1,Tobias Weigel,1. FC Heidenheim,1,1,0,0
2,Cenny Neumann,1. FC Köln,83,2,0,0
3,Linus Guther,1. FC Union Berlin,14,1,0,0
4,Silas,1. FSV Mainz 05,342,8,1,0
5,Piero Hincapié,Bayer 04 Leverkusen,90,1,0,0
6,Pascal Groß,Borussia Dortmund,618,11,0,1
7,Aarón Anselmino,Borussia Dortmund,368,6,1,0
8,Luca Reggiani,Borussia Dortmund,245,5,1,0
9,Samuele Inacio,Borussia Dortmund,134,4,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  1. FC Heidenheim  —  SF sin salario:


,player,minutesPlayed
0,Leo Scienza,180
1,Tobias Weigel,1


  CG plantilla completa:


,player,player_norm
0,Adam Kölle,adam kolle
1,Adrian Beck,adrian beck
2,Arijon Ibrahimovic,arijon ibrahimovic
3,Benedikt Gimber,benedikt gimber
4,Budu Zivzivadze,budu zivzivadze
5,Christian Conteh,christian conteh
6,Diant Ramaj,diant ramaj
7,Eren Dinkçi,eren dinkci
8,Frank Feller,frank feller
9,Hennes Behrens,hennes behrens



  1. FC Köln  —  SF sin salario:


...

,player,player_norm
0,Alexander Nübel,alexander nubel
1,Ameen Al-Dakhil,ameen al dakhil
2,Angelo Stiller,angelo stiller
3,Atakan Karazor,atakan karazor
4,Badredine Bouanani,badredine bouanani
5,Bilal El Khannouss,bilal el khannouss
6,Chema Andrés,chema andres
7,Chris Führich,chris fuhrich
8,Dan-Axel Zagadou,dan axel zagadou
9,Deniz Undav,deniz undav



  VfL Wolfsburg  —  SF sin salario:


,player,minutesPlayed
0,Vaclav Cerny,46


  CG plantilla completa:


,player,player_norm
0,Aaron Zehnter,aaron zehnter
1,Adam Daghim,adam daghim
2,Andreas Skov Olsen,andreas skov olsen
3,Aster Vranckx,aster vranckx
4,Bence Dárdai,bence dardai
5,Christian Eriksen,christian eriksen
6,Cleiton,cleiton
7,Denis Vavro,denis vavro
8,Dzenan Pejcinovic,dzenan pejcinovic
9,Jakub Zielinski,jakub zielinski


In [25]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('silas', '1 fsv mainz 05'): ('silas katompa mvumpa', '1 fsv mainz 05')
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')

Matches manuales definidos: 1


In [26]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: silas (1 fsv mainz 05) → silas katompa mvumpa (1 fsv mainz 05)

Tras matches manuales: 457/494 (92.5%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

⚠️ Nombre con sufijo `_snapshot_20260428` para diferenciar del master definitivo
que se generará al cierre de la temporada (`master_germany_2526.csv`).

In [27]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_germany_2526_snapshot_20260428.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_germany_2526_snapshot_20260428.csv
   Jugadores totales:  494
   Con salario:        457
   Sin salario (NaN):  37
   Columnas:           122
